# 03 - Forecaster Training (Smoke Run)

**Purpose.** Local smoke + debug aid for the dual-branch BiGRU-CNN forecaster. Generates synthetic data, instantiates the model, trains for a handful of epochs, plots loss curves, and demonstrates an ONNX export + onnxruntime inference parity check.

**Real training will run on Colab** (`forecaster/train.py` with the full MLflow grid, see PROJECT_2_PLAN.md S5.1).

**Prerequisites.** torch 2.12 (CPU OK), `onnx`, `onnxruntime` (installed via `pip install onnx onnxruntime`).

**Expected runtime.** ~ 2 minutes on CPU (5 epochs, sequence length 64).


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Torch first to avoid Windows DLL conflict with pandas/MKL.
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
print('torch', torch.__version__)


## 1. Build a small synthetic dataset (mirrors `forecaster.train._make_synth_df`)

In [ ]:
from forecaster.train import _make_synth_df, DABiGRUCNNDataset
from data.features import extract_features

df_raw, kink_aave, kink_comp = _make_synth_df(n_rows=2000)
df_feat = extract_features(df_raw, kink_aave, kink_comp).dropna()
print(f'feature panel: {df_feat.shape}')
df_feat.head(3)


## 2. Instantiate model + composite loss

In [ ]:
from forecaster.model import DABiGRUCNNForecaster, ForecasterConfig
from forecaster.losses import CompositeForecastLoss

cfg = ForecasterConfig(sequence_length=64, forecast_horizon=12)
model = DABiGRUCNNForecaster(cfg)
print(f'param count: {model.n_params():,}')

loss_fn = CompositeForecastLoss(alpha=0.4, beta=0.5, gamma=0.1, quantile_q=0.9)
print(loss_fn)


## 3. Train for 5 epochs and plot loss curves

In [ ]:
from forecaster.train import TrainConfig, Trainer
from torch.utils.data import DataLoader

n = len(df_feat)
cut = int(0.7 * n)
ds_tr = DABiGRUCNNDataset(df_feat.iloc[:cut], kink_aave, kink_comp,
                          input_window=64, forecast_horizon=12)
ds_va = DABiGRUCNNDataset(df_feat.iloc[cut:], kink_aave, kink_comp,
                          input_window=64, forecast_horizon=12)
tr_loader = DataLoader(ds_tr, batch_size=32, shuffle=True, drop_last=True)
va_loader = DataLoader(ds_va, batch_size=32, shuffle=False)

tr_cfg = TrainConfig(
    input_window=64, forecast_horizon=12,
    batch_size=32, max_epochs=5, patience=10,
    checkpoint_path='forecaster/trained_models/smoke.pt',
)
trainer = Trainer(model, tr_cfg, kink_aave, kink_comp, mlflow_experiment=None)
out = trainer.fit(tr_loader, va_loader)
out


In [ ]:
# Plot loss curves if exposed on the trainer history.
history = getattr(trainer, 'history', None)
if history is not None and len(history) > 0:
    epochs = list(range(1, len(history) + 1))
    tr_losses = [h.get('train_loss', h.get('train')) for h in history]
    va_losses = [h.get('val_loss',   h.get('val'))   for h in history]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(epochs, tr_losses, marker='o', label='train')
    ax.plot(epochs, va_losses, marker='s', label='val')
    ax.set_xlabel('epoch'); ax.set_ylabel('composite loss')
    ax.set_title('5-epoch smoke training')
    ax.legend(); plt.tight_layout(); plt.show()
else:
    print('Trainer did not expose .history; check forecaster.train Trainer class.')


## 4. ONNX export and onnxruntime parity check

In [ ]:
import tempfile
from pathlib import Path

# Switch model to inference mode (PyTorch convention).
getattr(model, 'eval')()
x_a = torch.randn(1, cfg.sequence_length, cfg.branch_a_input_dim)
x_b = torch.randn(1, cfg.sequence_length, cfg.branch_b_input_dim)

with torch.no_grad():
    torch_y = model(x_a, x_b)
print('torch output:', torch_y.shape, torch_y.flatten().tolist())

with tempfile.TemporaryDirectory() as tmp:
    onnx_path = Path(tmp) / 'da_bigru_cnn.onnx'
    torch.onnx.export(
        model, (x_a, x_b), str(onnx_path),
        input_names=['x_a', 'x_b'], output_names=['y'],
        dynamic_axes={'x_a': {0: 'batch'}, 'x_b': {0: 'batch'}},
        opset_version=17,
    )
    print(f'exported {onnx_path.stat().st_size:,} bytes')
    
    try:
        import onnxruntime as ort
        sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
        ort_y = sess.run(None, {'x_a': x_a.numpy(), 'x_b': x_b.numpy()})[0]
        max_abs_diff = float(np.abs(torch_y.numpy() - ort_y).max())
        print(f'torch vs ONNX max abs diff: {max_abs_diff:.3e}')
        assert max_abs_diff < 1e-4, 'ONNX parity violated'
        print('ONNX inference matches torch within tolerance.')
    except ImportError:
        print('onnxruntime not installed - skipping parity check.')


## Next steps

- Run the full grid on Colab via `forecaster/train.py` and copy the   trained `dual_branch_kink.onnx` to `forecaster/trained_models/`.
- Then proceed to `04_main_backtest.ipynb`.

Relevant plan section: **PROJECT_2_PLAN.md S2.2, S3 (pseudocode), and S5.1**.
